### Ejercicio 3

```c
double Riemann_Zeta(double s, int k) {
    double result = 0.0;

    for (int i = 1; i < k; i++)
        for (int j = 1; j < k; j++)
            result += (2*(i&1)-1)/pow(i+j, s);
            // i&1 --> 1 si i impar y 0 si es par
            // 2*(i&1)-1 --> (+1) impar o (-1) par --> (-1)^(i+1)
            // pow(i+j, s) --> (i+j)^s
    return result*pow(2, s); // multiplica por 2^s
}

for (unsigned int k = 0; k < N; k++)
    X[k] = Riemann_Zeta(2, k); // = pi^2/6

#### (i) Parallelize this loop using OpenMP. Are there any data dependencies or shared variables?

```c
#pragma omp parallel for
for (unsigned int k = 0; k < N; k++)
    X[k] = Riemann_Zeta(2, k);
```
En la linea 
```c 
X[k] = Riemann_Zeta(2, k);
``` 
la variable X es compartida y escribe cada iteración en su propia posición del arreglo X[k], no necesita el resultado de otra iteración para ejecutarse, no hay dependencia. 
Mientras que en la linea 
```c 
return result*pow(2, s);
``` 
la variable result es local a cada llamada, cada hilo trabaja con su propia copia, no hay dependencia.  

#### (ii) Elaborate on the load balancing of the individual threads.

La función 
```c
Riemann_Zeta(2, k)
```
contiene un doble for, por lo tanto cada iteración $k$ realiza $k^2$ cantidad de operaciones. Debido a que OpenMP divide el loop en bloques iguales, los primeros hilos termiran rapidamente mientras los ultimos, que trabajaran mucho mas tardaran mas tiempo. 

El programa ejecutado termina cuando todos los hilos han terminado, entonces el tiempo total lo dicta el hilo mas lento. Teniendo todo esto en cuenta, podemos deducir que existe un gran desbalance de carga entre los hilos. 

#### (iii) Discuss different scheduling modes and the corresponding runtimes.

scheduling es la estrategia que usa OpenMP para repartir las iteraciones del loop entre los hilos, exiten varios modos:

```c
schedule(static)
```
Separa las iteraciones en bloques iguales y contiguos, asignados antes de que el programa empiece a ejecutar (asignación estatica), cada hilo sabe qué iteraciones le tocan desde el inicio.

Para este problema, se le asignan iteraciones mas pequeñas a unos hilos, mientras que otras muy grandes a otros, lo que nos devuleve al problema planteado en la respuesta (ii). 

```c
schedule(static, 1)
```
Reparte las iteraciones en un round-robin entre los hilos, asignado antes de ejecutar el programa. Cada hilo tiene una mezcla de iteraciones pequeñas y grandes, por lo que deberia ser mas equitativo, pero a pesar de ser mejor que schedule(static), los ultimos hilos siguen recibiendo las iteraciones mas pesadas y costosas. 

```c
schedule(dynamic, 1)
```
No existe asignación previa, cada hilo cuando termina su tarea actual, va a la cola y pide la siguiente iteración disponible. Entonces, los hilos que terminan rápido toman más trabajo automáticamente, generando un balance casi perfecto, pero debido a que cada vez que un hilo pide trabajo hay una sincronización con el scheduler, esto genera un gran costo en tiempo (overhead), ya que con (N-1) iteraciones, se realizan (N-1) sincronizaciones. 

```c
schedule(guided)
```
Funciona de manera similar a dynamic, pero en lugar de asignar siempre la misma cantidad de iteraciones por hilo, el tamaño de cada bloque de trabajo va decreciendo a medida que el programa avanza. Al inicio de la ejecución se asignan bloques grandes a cada hilo, entonces a medida que quedan menos iteraciones los bloques se vuelven pequeños, esto asegura el balance entre los hilos al final de la ejecución. El tamaño de cada bloque se calcula con la formula: 

$b = \frac{n^\circ iteraciones penientes}{n^\circ hilos disponibles}$. 

Debido a que en este problema la carga crece de la forma $k^2$, parece ser perfecto, ya que las iteraciones más costosas están al final del rango, los bloques pequeños que asigna en esa etapa permiten distribuir ese trabajo pesado de forma más equitativa entre todos los hilos, logrando un buen balance y con un overhead mucho menor que dynamic. 

Entonces, el código paralelizado debria quedar:
```c
#pragma omp parallel for schedule(guided)
for (unsigned int k = 0; k < N; k++)
    X[k] = Riemann_Zeta(2, k);

```

### Ejercicio 5

#### (i) Máximo común divisor

Esta operación es asociativa $gcd(a, gcd(b, c)) = gcd(gcd(a, b), c)$, tiene un elemento neutro $gcd(a, 0) = a$ y es conmutativa $gcd(a,b) = gcd(b,a)$. Por lo tanto, si se puede utilizar para reducción paralela en OpenMP. 

#### (ii) Producto matricial de matrices 2×2

Esta operación es asociativa $(A B) C = A (B C)$ y tiene el elemento neutro $A I = I A = A$, donde $I$ es la matriz identidad que contiene unos en la diagonal, pero no es conmutativa $A B ≠ B A$, las reducciones paralelas estándar asumen que el orden no importa, por lo que se necesita una reducción personalizada. Por lo tanto, si se puede utilizar para reducción paralela en OpenMP, pero requiere cuidado con el orden.

#### (iii) Producto de números complejos

Esta operación es asociativa $(z₁ z₂) z₃ = z₁ (z₂ z₃)$, tiene un elemento neutro $z a = z$ con $a = (1 + 0i)$ y es conmutativa $z₁ z₂ = z₂ z₁$. Por lo tanto, si se puede utilizar para reducción paralela en OpenMP. 

#### (iv) Producto de cuaterniones

Esta operación es asociativa $(q₁ q₂) q₃ = q₁ (q₂ q₃)$, tiene un elemento neutro $z a = z$ con $a = (1 + 0i + 0j + 0k)$, pero no es conmutativa $q₁ q₂ ≠ q₂  q₁$. Por lo tanto, si se puede utilizar 
para reducción paralela en OpenMP, pero requiere cuidado con el orden.

#### (v) Media incremental por pares

Esta operación no es ni asociativa ni tiene un elemento neutro, esto se puede demostrar reemplazando valores en estas definiciones y se obtendra una desigualdad siempre. Por lo tanto, no se puede utilizar para reducción paralela en OpenMP. 